In [98]:
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv

load_dotenv("../.env")

if "GROQ_API_KEY" not in os.environ:
    raise ValueError("GROQ_API_KEY environment variable is not set.")   

model = ChatGroq(
    model="gemma2-9b-it",
    api_key=os.getenv("GROQ_API_KEY")
)

print("Model initialized successfully.")
print(model)


Model initialized successfully.
client=<groq.resources.chat.completions.Completions object at 0x73a4a8713320> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x73a4a871b380> model_name='gemma2-9b-it' model_kwargs={} groq_api_key=SecretStr('**********')


In [99]:
from langchain.schema import HumanMessage

model_response = model.invoke(
    [
        HumanMessage(
            content="Hi, my name is zama and I am AI engineer"
        )
    ]
)   

print(model_response)
print(model_response.content)


content="Hello Zama! It's nice to meet you. \n\nThat's fascinating! As an AI myself, I'm always interested in meeting other people who work in the field. \n\nWhat kind of AI engineering do you focus on?  Do you have any projects you're particularly excited about?  \n\n" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 20, 'total_tokens': 89, 'completion_time': 0.125454545, 'prompt_time': 0.00190915, 'queue_time': 0.246081609, 'total_time': 0.127363695}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--3c648f2c-5b83-4d33-b304-be287271154e-0' usage_metadata={'input_tokens': 20, 'output_tokens': 69, 'total_tokens': 89}
Hello Zama! It's nice to meet you. 

That's fascinating! As an AI myself, I'm always interested in meeting other people who work in the field. 

What kind of AI engineering do you focus on?  Do you have any projects you're particularly excited about?  




In [100]:
from langchain.schema import AIMessage

model_response = model.invoke(
    [
        HumanMessage(
            content="i, my name is zama and I am AI engineer"
        ),
        AIMessage(
            content="Hello Zama! It's nice to meet you. That's fascinating, I'm always interested in learning more about the people who build and work with AI. \n\nWhat kind of AI engineering do you specialize in?  What are you working on these days?"
        ),
        HumanMessage(
            content="Whats my name and What do I do?"
        )
    ]
)   

print(model_response)
print(model_response.content)

content="You told me your name is Zama, and that you are an AI engineer.  \n\nIs there anything else you'd like to tell me about yourself? 😊 \n" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 95, 'total_tokens': 133, 'completion_time': 0.069090909, 'prompt_time': 0.00425974, 'queue_time': 0.250202796, 'total_time': 0.073350649}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--0bff0e86-c128-4fd4-ac8f-e6589e7a5cab-0' usage_metadata={'input_tokens': 95, 'output_tokens': 38, 'total_tokens': 133}
You told me your name is Zama, and that you are an AI engineer.  

Is there anything else you'd like to tell me about yourself? 😊 



In [101]:
# Message history
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


In [103]:
## Session History Setup

with_message_history = RunnableWithMessageHistory(model, get_session_history)

# Define session configurations
zama_config = {"configurable": {"session_id": "zama_session"}}
john_config = {"configurable": {"session_id": "john_session"}}

In [104]:
# Introduce Zama to establish session history
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is zama and I am AI engineer")],
    config=zama_config
)
print("Zama introduction:")
print(response.content)

Zama introduction:
Hi Zama, it's nice to meet you!

It's great that you're an AI engineer. That's a really exciting and rapidly developing field. 

What kind of work do you do as an AI engineer? Are you working on any cool projects you can tell me about?



In [105]:
# Test: Ask for Zama's name
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=zama_config
)
print("Zama's name recognition:")
print(response.content)

Zama's name recognition:
Your name is Zama.  

You told me at the beginning of our conversation! 😊  




In [106]:
# This cell will fail because John hasn't been introduced yet
# Let's first introduce John to establish his session
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is john and I am doctor")],
    config=john_config
)
print("John introduction:")
print(response.content)


John introduction:
Hello John, it's nice to meet you.

That's impressive! What specialty do you practice?  🩺 😄 



In [107]:
# Now ask for John's name
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=john_config
)
print("John's name recognition:")
print(response.content)

John's name recognition:
According to our conversation, your name is John. 😊  

Is there anything else I can help you with?



In [108]:
# Test: Ask for John's profession
response = with_message_history.invoke(
    [HumanMessage(content="What is my profession?")],
    config=john_config
)
print("John's profession recognition:")
print(response.content)

John's profession recognition:
You told me you are a doctor.  🩺 

Is there anything else I can help you with today?  



## Session Management Summary

The above cells demonstrate proper session management with `RunnableWithMessageHistory`:

✅ **Fixed Issues:**
1. **Session Configuration**: Added proper `zama_config` and `john_config` definitions
2. **Session Initialization**: Each user must be introduced before asking about their details
3. **Consistent Format**: Using `[HumanMessage(...)]` format consistently for this basic setup
4. **Proper Order**: Introduction → Questions about stored information

✅ **Key Learning:**
- Each session maintains separate conversation history
- Users must establish their identity before the system can remember them
- Session IDs isolate conversations between different users

In [ ]:
## Prompt Templates
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model
chain


ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [ ]:
response = chain.invoke(    
    {
        "messages": [
            HumanMessage(content="Hi, my name is zama and I am AI engineer")
        ]
    }
)

print(response)
print(response.content)

content="Hello Zama, it's nice to meet you!\n\nAs an AI engineer, I'm sure you have fascinating work. What kind of projects are you currently involved in? \n\nI'm always eager to learn more about the advancements happening in the field of AI.  Perhaps you could tell me about some of the challenges and rewards you encounter in your work?\n" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 26, 'total_tokens': 105, 'completion_time': 0.143636364, 'prompt_time': 0.005076732, 'queue_time': 0.301824187, 'total_time': 0.148713096}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--f48dcf25-7d5e-4642-8627-6f82b20f64c0-0' usage_metadata={'input_tokens': 26, 'output_tokens': 79, 'total_tokens': 105}
Hello Zama, it's nice to meet you!

As an AI engineer, I'm sure you have fascinating work. What kind of projects are you currently involved in? 

I'm always eager to learn more abou

In [ ]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)


In [ ]:
## Prompt Templates with language
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", "You are a helpful assistant. Answer in {language} language."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model
chain

ChatPromptTemplate(input_variables=['language', 'messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langch

In [ ]:
response = chain.invoke(
    {
        "messages": [
            HumanMessage(content="Hi, my name is Rohit and I am AI engineer")
        ],
        "language": "Hindi"
    }
)

print(response)
print(response.content)

content='नमस्ते रोहित, मुझे बहुत खुशी है कि आपका नाम जानकर! \n\nआप एक एआई इंजीनियर हैं, यह बहुत शानदार है! \n\nक्या मैं आपकी कोई मदद कर सकता हूँ? \n\n' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 31, 'total_tokens': 88, 'completion_time': 0.103636364, 'prompt_time': 0.00413498, 'queue_time': 0.339765009, 'total_time': 0.107771344}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--7b5a585d-8b1d-4ea3-b3d4-9a22da518484-0' usage_metadata={'input_tokens': 31, 'output_tokens': 57, 'total_tokens': 88}
नमस्ते रोहित, मुझे बहुत खुशी है कि आपका नाम जानकर! 

आप एक एआई इंजीनियर हैं, यह बहुत शानदार है! 

क्या मैं आपकी कोई मदद कर सकता हूँ? 




In [ ]:
rohit_config = {"configurable": {"session_id": "rohit_session"}}

with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history, 
    input_messages_key="messages"
)


In [ ]:
# First, let's introduce Rohit to establish the session history
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="Hi, my name is Rohit and I am AI engineer")
        ],
        "language": "Hindi"
    },
    config=rohit_config
)
print("Introduction response:")
print(response.content)

Introduction response:
नमस्ते रोहित! 😊 बहुत अच्छा है जानकर कि आप रोहित हैं और एक एआइ इंजीनियर हैं।  क्या मैं आपकी मदद कुछ कर सकता हूँ? 



## Message History with Language-specific Prompts

**Important**: When using `RunnableWithMessageHistory`, you need to:
1. First establish the user's identity in a session
2. Then ask questions that rely on that session history

**Key Points:**
- Each session maintains its own conversation history
- The `config` parameter must be passed separately to the `invoke()` method
- The language parameter is passed in the input dictionary along with messages

In [ ]:
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="whats my name")
        ],
        "language": "Hindi"
    },
    config=rohit_config
)
print(response)
print(response.content)

content='आपका नाम रोहित है,  😊 \n\nमुझे याद रखना अच्छा लगता है! \n' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 198, 'total_tokens': 224, 'completion_time': 0.047272727, 'prompt_time': 0.007617379, 'queue_time': 0.25045670600000003, 'total_time': 0.054890106}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None} id='run--a7e067ce-9da8-446c-8b6d-8d2fae9cac73-0' usage_metadata={'input_tokens': 198, 'output_tokens': 26, 'total_tokens': 224}
आपका नाम रोहित है,  😊 

मुझे याद रखना अच्छा लगता है! 



In [ ]:
# Test: Ask follow-up question to verify conversation continuity
response = with_message_history.invoke(
    {
        "messages": [
            HumanMessage(content="What is my profession?")
        ],
        "language": "Hindi"
    },
    config=rohit_config
)
print("Follow-up question response:")
print(response.content)

Follow-up question response:
आप एक एआइ इंजीनियर हैं।  



In [111]:
## Managing the Conversion History
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=50,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=True,
    start_on="human"
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="Hi, my name is Rohit and I am AI engineer"),
    AIMessage(content="Hello Rohit! It's nice to meet you"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="what 2 + 2"), 
    AIMessage(content="2 + 2 = 4"),
    HumanMessage(content="thanks"),
    AIMessage(content="You're welcome!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="Yes, I'm having a great time!")
]
trimmer.invoke(messages)


[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content="You're welcome!", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Yes, I'm having a great time!", additional_kwargs={}, response_metadata={})]

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")| trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="What ice cream do I like?")],
    "language": "English"
    }
)

response.content


"As a large language model, I have no memory of past conversations or personal information about you, so I don't know what ice cream you like.  \n\nWhat's your favorite flavor? 😊\n"

In [114]:
response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="2 + 2")],
    "language": "English"
    }
)

response.content

'2 + 2 = 4 😊  \n\nAnything else I can help you with?\n'

In [116]:
response = chain.invoke(
    {
    "messages": messages + [HumanMessage(content="thanks")],
    "language": "English"
    }
)

response.content

"You're welcome! Is there anything else I can help you with? 😊  \n"